# Voix SIRA : fine-tune de Piper sur Colab

On part de la voix française `fr_FR-siwis-medium` (déjà entraînée pendant des jours) et on l'adapte à la voix enregistrée.
Le modèle apprend seulement le **timbre, l'accent et le rythme** de la lectrice : il garde la façon de prononcer le français.

**Avant de lancer :**
1. Menu **Exécution → Modifier le type d'exécution → GPU T4**.
2. Mets `dataset_piper_sira.zip` (préparé sur le PC : `piper_dataset.py decouper`, puis `piper_dataset.py emballer`) dans ton Google Drive, dossier **`SIRA_voix/`**.
3. **Exécution → Tout exécuter**. Compte environ 1 h pour 100 phrases (T4 gratuite). Garde l'onglet ouvert.

À la fin, tu trouves dans `Drive/SIRA_voix/resultat/` : le modèle `fr_FR-sira-medium.onnx` (+ `.json`) et les échantillons d'écoute.

## 1. GPU et Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/SIRA_voix'
import os; os.makedirs(f'{DRIVE}/resultat', exist_ok=True)
print(os.listdir(DRIVE))

## 2. Installer l'entraîneur Piper (version 1.8.0, figée pour que le notebook reste reproductible)

In [ ]:
!pip -q install "piper-tts[train]==1.8.0"
# Le paquet n'embarque pas la source de l'alignement monotone : on la compile depuis le dépôt, même version.
import pathlib, piper
MA = pathlib.Path(piper.__file__).parent / 'train/vits/monotonic_align'
(MA / 'monotonic_align').mkdir(exist_ok=True)
!wget -q -O {MA}/core.pyx https://raw.githubusercontent.com/OHF-Voice/piper1-gpl/v1.8.0/src/piper/train/vits/monotonic_align/core.pyx
!cd {MA} && cythonize -q -i core.pyx && mv core*.so monotonic_align/
from piper.train.vits.monotonic_align import maximum_path  # échoue ici si la compilation a raté
print('Piper prêt')

## 3. Données : les phrases découpées sur le PC

In [ ]:
!rm -rf /content/data && mkdir -p /content/data && unzip -q -o {DRIVE}/dataset_piper_sira.zip -d /content/data
import wave, glob
rows = [l.split('|', 1) for l in open('/content/data/metadata.csv', encoding='utf-8').read().splitlines() if '|' in l]
total = 0
for pid, _ in rows:
    with wave.open(f'/content/data/wavs/{pid}.wav') as w: total += w.getnframes() / w.getframerate()
print(f'{len(rows)} phrases, {total/60:.1f} min de voix')
for pid, txt in rows[:3]: print(' ', pid, '|', txt)

## 4. Voix de départ : checkpoint `fr_FR-siwis-medium` (846 Mo)

In [ ]:
BASE = '/content/siwis.ckpt'
!test -f {BASE} || wget -q --show-progress -O {BASE} "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/fr/fr_FR/siwis/medium/epoch%3D3304-step%3D2050940.ckpt"
# Vérification : les poids de siwis doivent presque tous rentrer dans le modèle Piper 1.8.
import torch
from piper.train.vits.lightning import VitsModel
sd_old = torch.load(BASE, map_location='cpu', weights_only=False)['state_dict']
sd_new = VitsModel(mos_metric=None).state_dict()
ok = sum(1 for k, v in sd_old.items() if k in sd_new and sd_new[k].shape == v.shape)
print(f'{ok} / {len(sd_new)} paramètres repris de siwis')
assert ok > 0.9 * len(sd_new), 'Checkpoint incompatible : ne pas lancer l’entraînement'
del sd_old, sd_new

## 5. Entraînement

`EPOQUES` : nombre de passages sur toutes les phrases. Avec ~100 phrases, 600 époques ≈ 3 500 pas ≈ 1 h sur T4.
Avec les 13 pistes (628 phrases), une époque est 6 fois plus longue : 200 à 300 époques suffiront.

In [ ]:
EPOQUES = 600
import glob, subprocess

def entrainer(csv, dossier, epoques):
    os.makedirs(dossier, exist_ok=True)
    reprise = sorted(glob.glob(f'{dossier}/lightning_logs/version_*/checkpoints/last.ckpt'))
    depart = f'--ckpt_path {reprise[-1]}' if reprise else f'--model.warmstart_ckpt {BASE}'
    cmd = (f'python -m piper.train fit --data.voice_name sira --data.csv_path {csv} --data.audio_dir /content/data/wavs '
           f'--data.espeak_voice fr --model.sample_rate 22050 --data.cache_dir {dossier}/cache '
           f'--data.config_path {dossier}/config.json --data.batch_size 16 --data.validation_split 0.05 '
           f'--data.num_test_examples 2 --model.mos_metric none --trainer.max_epochs {epoques} '
           f'--trainer.check_val_every_n_epoch 50 --trainer.default_root_dir {dossier} {depart}')
    print(cmd)
    get_ipython().system(cmd)
    return sorted(glob.glob(f'{dossier}/lightning_logs/version_*/checkpoints/last.ckpt'))[-1]

CKPT = entrainer('/content/data/metadata.csv', '/content/sira', EPOQUES)
print('Dernier checkpoint :', CKPT)

## 6. Export en modèle Piper (`.onnx`), le format que lit le service vocal SIRA

In [ ]:
def exporter(ckpt, dossier, nom):
    get_ipython().system(f'python -m piper.train.export_onnx --checkpoint {ckpt} --output-file {dossier}/{nom}.onnx')
    get_ipython().system(f'cp {dossier}/config.json {dossier}/{nom}.onnx.json')
    return f'{dossier}/{nom}.onnx'

MODELE = exporter(CKPT, '/content/sira', 'fr_FR-sira-medium')
!wget -q -nc -P /content/siwis "https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium/fr_FR-siwis-medium.onnx" "https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium/fr_FR-siwis-medium.onnx.json"

## 7. Écouter

- **Vraie voix** : un enregistrement d'origine, pour comparer le timbre.
- **Phrases jamais lues** : tirées des pistes 3 à 13 (non enregistrées). C'est le vrai test : c'est ce que SIRA dira tous les jours.
- **siwis** : la voix actuelle de SIRA, pour mesurer le changement.

In [ ]:
from piper import PiperVoice
from IPython.display import Audio, HTML, display

NOUVELLES = [
    "Bonjour, je suis SIRA. Où veux-tu aller aujourd'hui ?",
    "Prochain arrêt : Siporex.",
    "Prépare-toi, tu descends au prochain arrêt : Danga.",
    "Attention, accident signalé à la gare Sud. Je te propose un autre chemin.",
    "Forte pluie annoncée vers dix-sept heures. La route du pont HKB inonde souvent.",
    "Grève des gbaka à Anyama ce matin. Prévois un autre transport.",
    "Je n'ai pas bien entendu. Tu peux répéter un peu plus fort ?",
    "Si tu veux, je peux te réveiller avant ton arrêt.",
    "Tu veux partir maintenant ou plus tard ?",
]
DEJA_LUES = rows[:2]   # phrases du dataset : vraie voix contre modèle

def dire(modele, texte, chemin):
    with wave.open(chemin, 'wb') as w: PiperVoice.load(modele).synthesize_wav(texte, w)
    return chemin

def ecouter(modele, dossier, etiquette):
    os.makedirs(dossier, exist_ok=True)
    display(HTML(f'<h3>{etiquette}</h3>'))
    for pid, txt in DEJA_LUES:
        display(HTML(f'<b>Déjà lue</b> : {txt}<br>vraie voix puis modèle'))
        display(Audio(f'/content/data/wavs/{pid}.wav')); display(Audio(dire(modele, txt, f'{dossier}/{pid}.wav')))
    for i, txt in enumerate(NOUVELLES):
        display(HTML(f'<b>Jamais lue</b> : {txt}')); display(Audio(dire(modele, txt, f'{dossier}/nouvelle_{i+1:02d}.wav')))

ecouter(MODELE, '/content/echantillons/sira', f'Voix SIRA ({len(rows)} phrases)')
ecouter('/content/siwis/fr_FR-siwis-medium.onnx', '/content/echantillons/siwis', 'Voix actuelle (siwis)')

## 8. Sauvegarder dans le Drive

In [ ]:
!cp /content/sira/fr_FR-sira-medium.onnx /content/sira/fr_FR-sira-medium.onnx.json {DRIVE}/resultat/
!cp {CKPT} {DRIVE}/resultat/sira_last.ckpt
!cd /content && zip -q -r {DRIVE}/resultat/echantillons.zip echantillons
!ls -la {DRIVE}/resultat

## 9. (Optionnel, +1 h) Courbe de progression : 50 phrases contre toutes les phrases

On réentraîne avec seulement la **moitié** des phrases, mêmes réglages, puis on écoute les mêmes phrases jamais lues.
L'écart entre « 50 » et « 100 » montre ce que chaque piste en plus apporte. Avec 628 phrases, l'écart continue dans le même sens.

In [ ]:
moitie = '/content/data/metadata_moitie.csv'
open(moitie, 'w', encoding='utf-8').write(''.join(f'{p}|{t}\n' for p, t in rows[: len(rows) // 2]))
CKPT50 = entrainer(moitie, '/content/sira50', EPOQUES)
MODELE50 = exporter(CKPT50, '/content/sira50', 'fr_FR-sira50-medium')
ecouter(MODELE50, '/content/echantillons/sira50', f'Voix SIRA ({len(rows) // 2} phrases seulement)')
!cd /content && zip -q -r {DRIVE}/resultat/echantillons.zip echantillons